# Batch API corregido para tu modelo fine-tuned

Esta versión usa un formato de inferencia más parecido al entrenamiento:

- `system` con las clases
- `user` = solo el texto
- salida esperada = texto plano con etiquetas separadas por coma
- cadena vacía = sin etiquetas

In [ ]:
# Si hace falta:
# !pip install -q openai pandas

In [1]:
import os
import json
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from openai import OpenAI

## Configuración

In [ ]:

MODEL_NAME = 'gpt-4.1-mini'
INPUT_CSV = 'DNU_Data/muestra_100.csv'
TEXT_COLUMN = None
MAX_ROWS = None

REQUESTS_JSONL = 'outs_temp/batch_requests_labels_plaintext.jsonl'
RESULTS_JSONL = 'outs_temp/batch_results_labels_plaintext.jsonl'
OUTPUT_CSV = 'outs_temp/muestra_con_batch_gpt_corregido.csv'

COMPLETION_WINDOW = '24h'
MAX_TOKENS = 20
TEMPERATURE = 0

PRICE_INPUT_PER_1M_SYNC = 3.00
PRICE_OUTPUT_PER_1M_SYNC = 6.00

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('No encuentro OPENAI_API_KEY en variables de entorno.')

## Cargar dataset

In [5]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas totales: {len(df)}')
print('Columnas:')
print(list(df.columns))
df.head(3)

Filas totales: 100
Columnas:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels', 'HATEFUL']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels,HATEFUL
0,Sat Mar 06 23:16:38 +0000 2021,1368339770686996480,1368339770686996480,Extranjeros participan de los destrozos realiz...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0
1,Fri Mar 05 17:33:11 +0000 2021,1367890949792272391,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367887e+18,1.367887e+18,142093384.0,NaN,...,0,0,0,0,0,0,0,[],0,0
2,Fri Mar 05 22:02:06 +0000 2021,1367958627739258880,1367958627739258880,La diferencia con los kirchneristas ya no es s...,"<a href=""http://twitter.com/download/iphone"" r...",True,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,[],0,0


In [6]:
def detect_text_column(dataframe: pd.DataFrame) -> str:
    candidates = ['text', 'texto', 'tweet', 'contenido', 'content', 'body', 'comment', 'comentario', 'post', 'mensaje', 'message']
    lower_map = {c.lower(): c for c in dataframe.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    object_cols = [c for c in dataframe.columns if dataframe[c].dtype == 'object']
    if len(object_cols) == 1:
        return object_cols[0]
    raise ValueError('No pude detectar automáticamente la columna de texto. Define TEXT_COLUMN manualmente.')

text_col = TEXT_COLUMN if TEXT_COLUMN is not None else detect_text_column(df)
work_df = df.copy()
if MAX_ROWS is not None:
    work_df = work_df.head(MAX_ROWS).reset_index(drop=True)

print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

Columna de texto: text
Filas a procesar: 100


## Prompt corregido

In [7]:
SYSTEM_MESSAGE = """You are trained to analyze hate speech in text. Classify the text with one or more of the following labels: WOMEN, LGBTI, RACISM, CLASS, POLITICS, DISABLED, APPEARENCE, CRIMINAL, CALLS. Return only the matching labels separated by commas. If no label applies, return an empty string."""

USER_TEMPLATE = "{text}"

## Prueba sincrónica rápida antes del batch

In [11]:
import random

def test_single_call(text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': USER_TEMPLATE.format(text=text)}
        ]
    )
    return response.choices[0].message.content


# --------- NUEVO BLOQUE ---------

SEED = 83
N_SAMPLES = 5

random.seed(SEED)

indices = random.sample(range(len(work_df)), min(N_SAMPLES, len(work_df)))

for i in indices:
    txt = str(work_df[text_col].iloc[i]) if pd.notna(work_df[text_col].iloc[i]) else ''
    out = test_single_call(txt)

    print('=' * 80)
    print('fila', i)
    print('texto:', repr(txt[:250]))
    print('salida:', repr(out))

fila 63
texto: '@DeSangreTana Eso ya lo contempla la ley: \nhttps://t.co/k8SliEsnVD'
salida: ''
fila 58
texto: '@todonoticias Cuando la gente lee esto solo piensa en gente marroncita, porque para ellos un blanquito europeo o ya… https://t.co/FOLZoLDUIX'
salida: ''
fila 11
texto: '@wekaweko1 Son mano de obra barata, extranjeros delincuentes, para causar terror, eso hizo Masturro. Zonas liberada… https://t.co/jt3GgmRMun'
salida: ''
fila 16
texto: '@todonoticias Este gobierno es nefasto. No cuida a la gente. Cómo no tenemos delincuencia, van a dejar entrar a ext… https://t.co/6tD3Z2687C'
salida: ''
fila 48
texto: '@UltimaHoracom "Delincuentes asaltados por delincuentes" \nOsino\n"Entrega guiada"'
salida: ''


In [12]:
def debug_single_call(text: str):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        max_tokens=30,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": text}
        ],
        logprobs=True,
        top_logprobs=5
    )
    return response

textos_test = [
    "Las mujeres no deberían trabajar, solo servir a sus maridos",
    "Los inmigrantes negros son todos delincuentes",
    "Los gays son una enfermedad",
    "Los pobres de la villa arruinan el país",
    "Los políticos son una basura y habría que matarlos"
]

for t in textos_test:
    r = debug_single_call(t)
    print("=" * 80)
    print("TEXTO:", t)
    print("SALIDA:", repr(r.choices[0].message.content))
    print("TOKENS COMPLETION:", r.usage.completion_tokens)
    print("FINISH:", r.choices[0].finish_reason)

TEXTO: Las mujeres no deberían trabajar, solo servir a sus maridos
SALIDA: 'WOMEN'
TOKENS COMPLETION: 3
FINISH: stop
TEXTO: Los inmigrantes negros son todos delincuentes
SALIDA: 'RACISM'
TOKENS COMPLETION: 3
FINISH: stop
TEXTO: Los gays son una enfermedad
SALIDA: 'LGBTI'
TOKENS COMPLETION: 3
FINISH: stop
TEXTO: Los pobres de la villa arruinan el país
SALIDA: ''
TOKENS COMPLETION: 0
FINISH: stop
TEXTO: Los políticos son una basura y habría que matarlos
SALIDA: ''
TOKENS COMPLETION: 0
FINISH: stop


In [13]:
from openai import OpenAI
import json

client = OpenAI()

BASE_MODEL = "gpt-4.1-mini"

SYSTEM_MESSAGE_BASE = """You are a classifier of xenophobic speech.

Your task is to determine whether a text contains xenophobic hate speech directed at foreigners, immigrants, migrants, or people of a particular nationality or national origin.

Label definitions:
- XENOPHOBIC: the text expresses hostility, contempt, discrimination, dehumanization, exclusion, blame, or hatred against foreigners, immigrants, migrants, or people of a specific nationality or national origin.
- NOT_XENOPHOBIC: the text does not contain xenophobic hate speech.

Output format:
Return only valid JSON in one line with this structure:
{"label":"XENOPHOBIC"}
or
{"label":"NOT_XENOPHOBIC"}"""

def classify_xenophobia(text: str):
    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        max_tokens=30,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE_BASE},
            {"role": "user", "content": text}
        ]
    )

    raw = response.choices[0].message.content

    try:
        parsed = json.loads(raw)
        label = parsed.get("label")
    except Exception:
        parsed = None
        label = None

    return {
        "raw_response": raw,
        "label": label,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens
    }

In [15]:
import random

SEED = 76
N_SAMPLES = 10

random.seed(SEED)
indices = random.sample(range(len(work_df)), min(N_SAMPLES, len(work_df)))

for i in indices:
    txt = str(work_df[text_col].iloc[i]) if pd.notna(work_df[text_col].iloc[i]) else ""
    out = classify_xenophobia(txt)

    print("=" * 80)
    print("fila", i)
    print("texto:", repr(txt[:300]))
    print("salida:", out["raw_response"])
    print("label:", out["label"])

fila 47
texto: '@EldiariodeLeuco Los delincuentes en este gobierno, pertenecen a la clase privilegiada'
salida: {"label":"NOT_XENOPHOBIC"}
label: NOT_XENOPHOBIC
fila 59
texto: '@JorgeFaurie error sería si lo hicieran sin darse cuenta, saben perfectamente lo q hacen, son delincuentes y quiere… https://t.co/ukbQSketZm'
salida: {"label":"NOT_XENOPHOBIC"}
label: NOT_XENOPHOBIC
fila 49
texto: 'Residentes extranjeros serán vacunados contra la COVID-19, afirmó Óscar Ugarte\n\nEl ministro de Salud indicó que com… https://t.co/jaqWvut7Wa'
salida: {"label":"NOT_XENOPHOBIC"}
label: NOT_XENOPHOBIC
fila 25
texto: '@szamhht Exacto, son DELINCUENTES'
salida: {"label":"NOT_XENOPHOBIC"}
label: NOT_XENOPHOBIC
fila 38
texto: '@MueroPorSalir @LunaDeCafe Es verdad, si en este sexenio solo son abrazos para los delincuentes y gas lacrimógeno p… https://t.co/L8ED8M4CgY'
salida: {"label":"NOT_XENOPHOBIC"}
label: NOT_XENOPHOBIC
fila 6
texto: '@MikeHam060 Fueron expulsados por tener antecedentes penal y gracias 